# GPU no Claude + API (texto, imagem, vídeo, fala, transcrição)

1. Ative a GPU: **Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU**.
2. Cole seu token do ngrok na célula **Configuração** (https://dashboard.ngrok.com/get-started/your-authtoken).
3. (Recomendado) Preencha `MCP_TOKEN` e `NGROK_DOMINIO` para a URL do conector não mudar quando o Colab reiniciar. O domínio grátis fica em https://dashboard.ngrok.com/domains.
4. Clique em **Ambiente de execução → Executar tudo**.
5. Copie as URLs que aparecerem:
   - `conector_claude` → Claude → Configurações → Conectores → Adicionar conector personalizado
   - `api` → API própria (`/texto`, `/imagem`, `/video`, `/fala`, `/transcrever`, `/pipeline`, `/tarefa/{id}`)
   - `openai_base_url` → qualquer app ou biblioteca compatível com OpenAI

⚠️ As URLs contêm um token secreto: não compartilhe. Confira os termos de uso do Colab e a licença de cada modelo.

Notebook completo: todo o código está aqui dentro, não precisa baixar nada do GitHub.

## 1. Instalar dependências

In [ ]:
!pip install -q "mcp>=1.10,<2" "pyngrok" "python-multipart" "huggingface_hub[hf_xet]" "transformers>=4.46" "diffusers>=0.33" "accelerate" "bitsandbytes" "sentencepiece" "ftfy" "imageio" "imageio-ffmpeg" "scipy" "numpy"

## 2. Código do servidor

In [ ]:
%%writefile motor.py
"""Carrega e roda modelos do Hugging Face na GPU, otimizado para velocidade.

Tipos suportados:
  texto       -> LLMs de chat (transformers AutoModelForCausalLM)
  imagem      -> texto para imagem (diffusers)
  video       -> texto para vídeo (diffusers)
  fala        -> texto para fala (pipeline text-to-speech)
  transcrever -> fala para texto (pipeline automatic-speech-recognition)
  qualquer outra tarefa do transformers.pipeline (ex.: translation, summarization,
  image-classification, object-detection...)
"""
import base64
import gc
import io
import os
import threading
import uuid
from collections import OrderedDict

# Download paralelo e rápido dos pesos (hf_xet).
os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")

PASTA_SAIDA = os.environ.get("PASTA_SAIDA", "saidas")
MAX_MODELOS = int(os.environ.get("MAX_MODELOS", "2"))
COMPILAR = os.environ.get("COMPILAR", "0") == "1"
# Limiar do cache entre passos nos modelos de vídeo (0 desliga; maior = mais rápido, menos fiel).
CACHE_VIDEO = float(os.environ.get("CACHE_VIDEO", "0.05"))
# LoRA de destilação para vídeo (ex.: CausVid), permite 4-8 passos com guidance 1.
LORA_VIDEO = os.environ.get("LORA_VIDEO", "")

PADROES = {
    "texto": os.environ.get("MODELO_TEXTO", "Qwen/Qwen2.5-1.5B-Instruct"),
    "imagem": os.environ.get("MODELO_IMAGEM", "stabilityai/sdxl-turbo"),
    "video": os.environ.get("MODELO_VIDEO", "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"),
    "fala": os.environ.get("MODELO_FALA", "facebook/mms-tts-por"),
    "transcrever": os.environ.get("MODELO_TRANSCRICAO", "openai/whisper-large-v3-turbo"),
}

TAREFAS_PIPELINE = {
    "fala": "text-to-speech",
    "transcrever": "automatic-speech-recognition",
}

_modelos = OrderedDict()  # repo_id -> {"tipo": ..., "obj": ...}
_trava = threading.Lock()  # uma geração por vez na GPU evita falta de memória
_configurado = False

os.makedirs(PASTA_SAIDA, exist_ok=True)


# ---------------------------------------------------------------- utilidades

def _torch():
    global _configurado
    import torch

    if not _configurado and torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.backends.cudnn.benchmark = True
        _configurado = True
    return torch


def _dispositivo():
    return "cuda" if _torch().cuda.is_available() else "cpu"


def _dtype():
    torch = _torch()
    if not torch.cuda.is_available():
        return torch.float32
    # bf16 só é rápido a partir da arquitetura Ampere (A100, L4, RTX 30xx); T4 usa fp16.
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


def _vram_gb():
    torch = _torch()
    if not torch.cuda.is_available():
        return 0
    return torch.cuda.get_device_properties(0).total_memory / 1024**3


def _liberar(repo_id):
    _modelos.pop(repo_id, None)
    gc.collect()
    torch = _torch()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def salvar_arquivo(dados: bytes, extensao: str) -> str:
    nome = f"{uuid.uuid4().hex}.{extensao}"
    with open(os.path.join(PASTA_SAIDA, nome), "wb") as f:
        f.write(dados)
    return nome


def _ler_audio(audio: str):
    """Aceita URL http(s), caminho de arquivo ou base64."""
    if audio.startswith(("http://", "https://")) or os.path.exists(audio):
        return audio
    if "," in audio and audio.startswith("data:"):
        audio = audio.split(",", 1)[1]
    return base64.b64decode(audio)


def _json_seguro(valor):
    """Converte saídas de pipelines (numpy, tensores, PIL) em algo serializável."""
    if isinstance(valor, dict):
        return {k: _json_seguro(v) for k, v in valor.items()}
    if isinstance(valor, (list, tuple)):
        return [_json_seguro(v) for v in valor]
    if hasattr(valor, "save") and hasattr(valor, "size"):  # imagem PIL
        buf = io.BytesIO()
        valor.save(buf, format="PNG")
        return {"imagem_base64": base64.b64encode(buf.getvalue()).decode()}
    if hasattr(valor, "tolist"):
        return valor.tolist()
    return valor


# ---------------------------------------------------------------- carregamento

def _carregar_texto(repo_id, quantizar_4bit):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    kwargs = {"torch_dtype": _dtype(), "device_map": "auto", "attn_implementation": "sdpa"}
    if quantizar_4bit:
        from transformers import BitsAndBytesConfig

        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=_dtype(), bnb_4bit_quant_type="nf4"
        )
    tokenizer = AutoTokenizer.from_pretrained(repo_id)
    modelo = AutoModelForCausalLM.from_pretrained(repo_id, **kwargs).eval()
    return {"tokenizer": tokenizer, "modelo": modelo}


def _carregar_imagem(repo_id):
    from diffusers import AutoPipelineForText2Image

    torch = _torch()
    pipe = AutoPipelineForText2Image.from_pretrained(repo_id, torch_dtype=_dtype())
    if _dispositivo() == "cuda" and _vram_gb() < 14 and "flux" in repo_id.lower():
        pipe.enable_model_cpu_offload()  # FLUX não cabe inteiro numa T4
    else:
        pipe.to(_dispositivo())
    pipe.set_progress_bar_config(disable=True)
    rede = getattr(pipe, "unet", None) or getattr(pipe, "transformer", None)
    if rede is not None and _dispositivo() == "cuda":
        rede.to(memory_format=torch.channels_last)
        if COMPILAR:
            # Primeira geração demora mais; as seguintes ficam ~1,5-2x mais rápidas.
            setattr(pipe, "unet" if hasattr(pipe, "unet") else "transformer",
                    torch.compile(rede, mode="max-autotune-no-cudagraphs"))
    if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
        pipe.vae.enable_tiling()
    # Aquecimento: a primeira geração real já sai na velocidade máxima.
    with torch.inference_mode():
        pipe(prompt="aquecimento", num_inference_steps=1, guidance_scale=0.0,
             width=512, height=512)
    return pipe


def _carregar_lora(pipe, lora):
    """lora: "dono/repo" ou "dono/repo:arquivo.safetensors". Funde nos pesos (sem custo extra)."""
    repo, _, arquivo = lora.partition(":")
    pipe.load_lora_weights(repo, weight_name=arquivo or None, adapter_name="rapido")
    pipe.fuse_lora()
    pipe.unload_lora_weights()


def _carregar_video(repo_id, lora=None):
    from diffusers import DiffusionPipeline

    torch = _torch()
    kwargs = {"torch_dtype": _dtype()}
    if "wan" in repo_id.lower():
        # O VAE do Wan precisa de fp32 para não gerar vídeo preto; é pequeno, quase não pesa.
        from diffusers import AutoencoderKLWan

        kwargs["vae"] = AutoencoderKLWan.from_pretrained(repo_id, subfolder="vae",
                                                         torch_dtype=torch.float32)
        if _dispositivo() == "cuda" and _vram_gb() < 24:
            # O codificador de texto (umt5-xxl, ~11 GB) estoura a RAM do Colab e não cabe com o
            # resto numa T4. Em 8 bits ele cai para ~6 GB e é carregado direto na GPU.
            try:
                from diffusers.quantizers import PipelineQuantizationConfig

                kwargs["quantization_config"] = PipelineQuantizationConfig(
                    quant_backend="bitsandbytes_8bit",
                    quant_kwargs={"load_in_8bit": True},
                    components_to_quantize=["text_encoder"],
                )
            except ImportError:
                print("diffusers sem PipelineQuantizationConfig; atualize para >=0.33.")
    pipe = DiffusionPipeline.from_pretrained(repo_id, **kwargs)
    if lora:
        _carregar_lora(pipe, lora)
    # Tudo na GPU é muito mais rápido; offload para a CPU só se realmente não couber.
    try:
        for nome, componente in pipe.components.items():
            # Componentes quantizados já estão na GPU e não podem ser movidos.
            if hasattr(componente, "to") and not getattr(componente, "is_loaded_in_8bit", False):
                componente.to(_dispositivo())
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        pipe.enable_model_cpu_offload()
    pipe.set_progress_bar_config(disable=True)
    if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
        pipe.vae.enable_tiling()
    if CACHE_VIDEO > 0 and hasattr(pipe, "transformer"):
        # Reaproveita cálculos entre passos parecidos: ~1,5-2x mais rápido, perda pequena.
        try:
            from diffusers import FirstBlockCacheConfig

            pipe.transformer.enable_cache(FirstBlockCacheConfig(threshold=CACHE_VIDEO))
        except Exception as e:
            print(f"Cache de vídeo indisponível nesta versão do diffusers: {e}")
    return pipe


def _carregar_pipeline(repo_id, tarefa):
    from transformers import pipeline

    return pipeline(
        tarefa,
        model=repo_id,
        torch_dtype=_dtype(),
        device=0 if _dispositivo() == "cuda" else -1,
    )


def carregar(repo_id: str | None = None, tipo: str = "texto", quantizar_4bit: bool = False,
             lora: str | None = None) -> str:
    repo_id = repo_id or PADROES.get(tipo)
    if not repo_id:
        raise ValueError(f"Informe o repo_id para o tipo '{tipo}'.")
    with _trava:
        if repo_id in _modelos:
            _modelos.move_to_end(repo_id)
            return f"{repo_id} já estava carregado ({tipo})."
        limite = MAX_MODELOS
        if tipo == "video" and _dispositivo() == "cuda" and _vram_gb() < 24:
            limite = 1  # vídeo precisa da GPU inteira numa T4/L4
        while _modelos and len(_modelos) >= limite:
            _liberar(next(iter(_modelos)))
        if tipo == "texto":
            obj = _carregar_texto(repo_id, quantizar_4bit)
        elif tipo == "imagem":
            obj = _carregar_imagem(repo_id)
        elif tipo == "video":
            obj = _carregar_video(repo_id, lora or LORA_VIDEO or None)
        else:
            obj = _carregar_pipeline(repo_id, TAREFAS_PIPELINE.get(tipo, tipo))
        _modelos[repo_id] = {"tipo": tipo, "obj": obj}
    return f"{repo_id} carregado como '{tipo}' em {_dispositivo()}."


def descarregar(repo_id: str) -> str:
    with _trava:
        if repo_id not in _modelos:
            return f"{repo_id} não estava carregado."
        _liberar(repo_id)
    return f"{repo_id} removido da memória."


def _obter(tipo, repo_id=None):
    repo_id = repo_id or next((r for r, m in reversed(_modelos.items()) if m["tipo"] == tipo), None)
    if repo_id is None or repo_id not in _modelos:
        carregar(repo_id, tipo)
        repo_id = repo_id or PADROES[tipo]
    _modelos.move_to_end(repo_id)
    return _modelos[repo_id]["obj"]


# ---------------------------------------------------------------- geração

def conversar(mensagens: list[dict], max_tokens: int = 512, temperatura: float = 0.7,
              modelo: str | None = None) -> str:
    torch = _torch()
    m = _obter("texto", modelo)
    tok, llm = m["tokenizer"], m["modelo"]
    if tok.chat_template:
        texto = tok.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    else:
        texto = "\n".join(f"{x['role']}: {x['content']}" for x in mensagens) + "\nassistant:"
    entradas = tok(texto, return_tensors="pt").to(llm.device)
    opcoes = {"max_new_tokens": max_tokens, "pad_token_id": tok.pad_token_id or tok.eos_token_id}
    if temperatura > 0:
        opcoes.update(do_sample=True, temperature=temperatura, top_p=0.95)
    else:
        opcoes["do_sample"] = False
    with _trava, torch.inference_mode():
        saida = llm.generate(**entradas, **opcoes)
    return tok.decode(saida[0, entradas["input_ids"].shape[1]:], skip_special_tokens=True)


def gerar_texto(prompt: str, sistema: str = "", **kwargs) -> str:
    mensagens = ([{"role": "system", "content": sistema}] if sistema else [])
    mensagens.append({"role": "user", "content": prompt})
    return conversar(mensagens, **kwargs)


def gerar_imagem(prompt: str, passos: int = 2, guidance: float = 0.0, largura: int = 768,
                 altura: int = 768, seed: int = -1, negativo: str = "",
                 modelo: str | None = None) -> bytes:
    torch = _torch()
    pipe = _obter("imagem", modelo)
    opcoes = {"prompt": prompt, "num_inference_steps": passos, "guidance_scale": guidance,
              "width": largura, "height": altura}
    if negativo and guidance > 1:
        opcoes["negative_prompt"] = negativo
    if seed >= 0:
        opcoes["generator"] = torch.Generator(_dispositivo()).manual_seed(seed)
    with _trava, torch.inference_mode():
        img = pipe(**opcoes).images[0]
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    return buf.getvalue()


PRESETS_VIDEO = {
    # quadros, passos, guidance, largura, altura
    "rapido": (17, 12, 5.0, 640, 368),
    "normal": (33, 20, 5.0, 832, 480),
    "qualidade": (81, 30, 5.0, 832, 480),
    "lora": (33, 6, 1.0, 832, 480),  # com LoRA de destilação (CausVid)
}


def gerar_video(prompt: str, quadros: int | None = None, passos: int | None = None,
                guidance: float | None = None, largura: int | None = None,
                altura: int | None = None, fps: int = 16, seed: int = -1,
                modelo: str | None = None, preset: str = "") -> str:
    """Gera um MP4 e devolve o nome do arquivo salvo em PASTA_SAIDA.

    preset: rapido | normal | qualidade | lora. Valores informados sobrescrevem o preset.
    """
    from diffusers.utils import export_to_video

    torch = _torch()
    preset = preset or ("lora" if LORA_VIDEO else "normal")
    q, p, g, lw, lh = PRESETS_VIDEO.get(preset, PRESETS_VIDEO["normal"])
    quadros, passos, guidance = quadros or q, passos or p, g if guidance is None else guidance
    largura, altura = largura or lw, altura or lh
    pipe = _obter("video", modelo)
    opcoes = {"prompt": prompt, "num_frames": quadros, "num_inference_steps": passos,
              "guidance_scale": guidance, "width": largura, "height": altura}
    if seed >= 0:
        opcoes["generator"] = torch.Generator(_dispositivo()).manual_seed(seed)
    with _trava, torch.inference_mode():
        frames = pipe(**opcoes).frames[0]
    nome = f"{uuid.uuid4().hex}.mp4"
    export_to_video(frames, os.path.join(PASTA_SAIDA, nome), fps=fps)
    return nome


def falar(texto: str, modelo: str | None = None) -> bytes:
    """Texto para fala. Devolve um WAV."""
    import numpy as np
    from scipy.io import wavfile

    pipe = _obter("fala", modelo)
    with _trava, _torch().inference_mode():
        saida = pipe(texto)
    audio = np.asarray(saida["audio"]).squeeze()
    buf = io.BytesIO()
    wavfile.write(buf, saida["sampling_rate"], (np.clip(audio, -1, 1) * 32767).astype(np.int16))
    return buf.getvalue()


def transcrever(audio: str, idioma: str | None = None, modelo: str | None = None) -> str:
    """Fala para texto. `audio` pode ser URL, caminho de arquivo ou base64."""
    pipe = _obter("transcrever", modelo)
    opcoes = {"chunk_length_s": 30, "batch_size": 8}
    if idioma:
        opcoes["generate_kwargs"] = {"language": idioma}
    with _trava, _torch().inference_mode():
        return pipe(_ler_audio(audio), **opcoes)["text"].strip()


_tarefas = {}  # id -> {"estado": "rodando" | "pronto" | "erro", "resultado": ..., "inicio": ...}


def em_segundo_plano(funcao, *args, **kwargs) -> str:
    """Roda uma função demorada numa thread e devolve o id para consultar depois."""
    import time

    tarefa_id = uuid.uuid4().hex[:12]
    _tarefas[tarefa_id] = {"estado": "rodando", "resultado": None, "inicio": time.time()}

    def rodar():
        try:
            _tarefas[tarefa_id].update(estado="pronto", resultado=funcao(*args, **kwargs))
        except Exception as e:
            _tarefas[tarefa_id].update(estado="erro", resultado=f"{type(e).__name__}: {e}")

    threading.Thread(target=rodar, daemon=True).start()
    return tarefa_id


def consultar(tarefa_id: str) -> dict:
    import time

    t = _tarefas.get(tarefa_id)
    if t is None:
        return {"estado": "desconhecida"}
    return {"estado": t["estado"], "resultado": t["resultado"],
            "segundos": int(time.time() - t["inicio"])}


def executar_pipeline(tarefa: str, repo_id: str, entrada, parametros: dict | None = None):
    """Roda qualquer tarefa do transformers.pipeline e devolve o resultado em JSON."""
    if repo_id not in _modelos:
        carregar(repo_id, tarefa)
    pipe = _modelos[repo_id]["obj"]
    with _trava, _torch().inference_mode():
        resultado = pipe(entrada, **(parametros or {}))
    return _json_seguro(resultado)


def status() -> dict:
    try:
        torch = _torch()
        if torch.cuda.is_available():
            livre, total = torch.cuda.mem_get_info()
            gpu = {"nome": torch.cuda.get_device_name(0),
                   "vram_livre_gb": round(livre / 1024**3, 1),
                   "vram_total_gb": round(total / 1024**3, 1)}
        else:
            gpu = {"nome": "sem GPU (CPU)"}
    except ImportError:
        gpu = {"nome": "torch não instalado"}
    return {"gpu": gpu,
            "modelos_carregados": {r: m["tipo"] for r, m in _modelos.items()},
            "padroes": PADROES,
            "max_modelos": MAX_MODELOS}


In [ ]:
%%writefile servidor.py
"""Expõe a GPU (Colab/Kaggle/RunPod/PC) como conector MCP do Claude e como API HTTP.

Tudo fica atrás de um token secreto que faz parte da URL:
  {base}/{TOKEN}/mcp                    -> conector personalizado do Claude
  {base}/{TOKEN}/api/...                -> API própria (texto, imagem, vídeo, fala, transcrição, pipeline)
  {base}/{TOKEN}/v1/...                 -> API compatível com OpenAI
  {base}/{TOKEN}/arquivos/{nome}        -> arquivos gerados (vídeos, áudios)
"""
import base64
import os
import secrets
import time
import uuid

from anyio import to_thread
from mcp.server.fastmcp import Audio, FastMCP, Image
from mcp.server.transport_security import TransportSecuritySettings
from starlette.requests import Request
from starlette.responses import FileResponse, JSONResponse, Response

import motor

# Defina MCP_TOKEN e NGROK_DOMINIO (domínio fixo grátis do ngrok) para a URL do conector
# continuar a mesma quando o Colab reiniciar.
TOKEN = os.environ.get("MCP_TOKEN") or secrets.token_urlsafe(24)
PORTA = int(os.environ.get("MCP_PORTA", "8000"))
NGROK_DOMINIO = os.environ.get("NGROK_DOMINIO", "")
BASE = f"/{TOKEN}"

mcp = FastMCP(
    "gpu-gratis",
    host="0.0.0.0",
    port=PORTA,
    streamable_http_path=f"{BASE}/mcp",
    stateless_http=True,
    # O ngrok muda o Host; a proteção fica por conta do token secreto na URL.
    transport_security=TransportSecuritySettings(enable_dns_rebinding_protection=False),
)

estado = {"url_publica": f"http://localhost:{PORTA}"}


def _url_arquivo(nome: str) -> str:
    return f"{estado['url_publica']}{BASE}/arquivos/{nome}"


async def _rodar(funcao, *args, **kwargs):
    """Roda a geração numa thread para não travar o servidor."""
    return await to_thread.run_sync(lambda: funcao(*args, **kwargs))


# ======================================================================= MCP

@mcp.tool()
async def carregar_modelo(repo_id: str = "", tipo: str = "texto", quantizar_4bit: bool = False,
                          lora: str = "") -> str:
    """Baixa e carrega um modelo do Hugging Face na GPU.

    tipo: texto | imagem | video | fala | transcrever | qualquer tarefa do transformers.pipeline
    (ex.: translation, summarization, image-classification). repo_id vazio usa o modelo padrão do tipo.
    quantizar_4bit=True permite LLMs maiores (ex.: 7B-8B numa T4).
    lora (vídeo): "dono/repo" ou "dono/repo:arquivo.safetensors" de uma LoRA de destilação.
    Vídeo carrega em segundo plano e devolve um id para consultar_tarefa.
    """
    if tipo == "video":
        tarefa_id = motor.em_segundo_plano(motor.carregar, repo_id or None, tipo, quantizar_4bit,
                                           lora or None)
        return f"Carregando em segundo plano. id da tarefa: {tarefa_id}. Consulte com consultar_tarefa."
    return await _rodar(motor.carregar, repo_id or None, tipo, quantizar_4bit, lora or None)


@mcp.tool()
async def descarregar_modelo(repo_id: str) -> str:
    """Remove um modelo da memória da GPU."""
    return await _rodar(motor.descarregar, repo_id)


@mcp.tool()
async def gerar_texto(prompt: str, sistema: str = "", max_tokens: int = 512,
                      temperatura: float = 0.7, modelo: str = "") -> str:
    """Gera texto com um LLM aberto (padrão Qwen2.5-1.5B-Instruct)."""
    return await _rodar(motor.gerar_texto, prompt, sistema, max_tokens=max_tokens,
                        temperatura=temperatura, modelo=modelo or None)


@mcp.tool()
async def gerar_imagem(prompt: str, passos: int = 2, guidance: float = 0.0, largura: int = 768,
                       altura: int = 768, seed: int = -1, negativo: str = "",
                       modelo: str = "") -> Image:
    """Gera uma imagem (padrão SDXL-Turbo).

    Modelos turbo/schnell: 1-4 passos e guidance 0. Modelos normais: 20-30 passos e guidance 5-7.
    """
    dados = await _rodar(motor.gerar_imagem, prompt, passos, guidance, largura, altura, seed,
                         negativo, modelo or None)
    return Image(data=dados, format="jpeg")


@mcp.tool()
async def gerar_video(prompt: str, preset: str = "", quadros: int = 0, passos: int = 0,
                      guidance: float = -1, largura: int = 0, altura: int = 0, seed: int = -1,
                      modelo: str = "") -> str:
    """Começa a gerar um vídeo curto (padrão Wan2.1 1.3B) em segundo plano e devolve um id.

    Vídeo leva minutos: depois chame consultar_tarefa(id) até o estado ser "pronto"; o resultado
    é o link do MP4. Na primeira vez o modelo também é baixado e carregado.
    preset: rapido (17 quadros, 12 passos, 640x368) | normal | qualidade | lora (6 passos, com LoRA).
    Valores 0/-1 usam os do preset.
    """
    def tarefa():
        nome = motor.gerar_video(prompt, quadros or None, passos or None,
                                 None if guidance < 0 else guidance, largura or None,
                                 altura or None, seed=seed, modelo=modelo or None, preset=preset)
        return _url_arquivo(nome)

    tarefa_id = motor.em_segundo_plano(tarefa)
    return f"Vídeo em geração. id da tarefa: {tarefa_id}. Consulte com consultar_tarefa."


@mcp.tool()
async def consultar_tarefa(tarefa_id: str) -> dict:
    """Consulta uma tarefa em segundo plano (vídeo ou carregamento de modelo).

    estado: rodando | pronto | erro. Se ainda estiver rodando, espere ~30-60 s e consulte de novo.
    """
    return motor.consultar(tarefa_id)


@mcp.tool()
async def gerar_audio(texto: str, modelo: str = "") -> Audio:
    """Transforma texto em fala (padrão MMS-TTS em português)."""
    return Audio(data=await _rodar(motor.falar, texto, modelo or None), format="wav")


@mcp.tool()
async def transcrever_audio(audio: str, idioma: str = "", modelo: str = "") -> str:
    """Transcreve áudio para texto (padrão Whisper large-v3-turbo). `audio`: URL ou base64."""
    return await _rodar(motor.transcrever, audio, idioma or None, modelo or None)


@mcp.tool()
async def executar_pipeline(tarefa: str, repo_id: str, entrada: str,
                            parametros: dict | None = None) -> dict:
    """Roda qualquer modelo do Hugging Face compatível com transformers.pipeline.

    Ex.: tarefa="translation", repo_id="Helsinki-NLP/opus-mt-en-pt", entrada="Hello world".
    """
    resultado = await _rodar(motor.executar_pipeline, tarefa, repo_id, entrada, parametros)
    return {"resultado": resultado}


@mcp.tool()
async def status() -> dict:
    """Mostra a GPU, a memória livre e os modelos carregados."""
    return motor.status()


# ================================================================ API própria

def _rota(caminho, metodos=("POST",)):
    return mcp.custom_route(f"{BASE}{caminho}", methods=list(metodos))


async def _corpo(request: Request) -> dict:
    try:
        return await request.json()
    except Exception:
        return {}


def _erro(e: Exception, codigo=500):
    return JSONResponse({"erro": str(e)}, status_code=codigo)


@_rota("/api/status", ("GET",))
async def api_status(request: Request):
    return JSONResponse(motor.status())


@_rota("/api/carregar")
async def api_carregar(request: Request):
    c = await _corpo(request)
    try:
        msg = await _rodar(motor.carregar, c.get("repo_id"), c.get("tipo", "texto"),
                           c.get("quantizar_4bit", False), c.get("lora"))
        return JSONResponse({"mensagem": msg})
    except Exception as e:
        return _erro(e)


@_rota("/api/descarregar")
async def api_descarregar(request: Request):
    c = await _corpo(request)
    return JSONResponse({"mensagem": await _rodar(motor.descarregar, c["repo_id"])})


@_rota("/api/texto")
async def api_texto(request: Request):
    c = await _corpo(request)
    try:
        if "mensagens" in c:
            texto = await _rodar(motor.conversar, c["mensagens"], c.get("max_tokens", 512),
                                 c.get("temperatura", 0.7), c.get("modelo"))
        else:
            texto = await _rodar(motor.gerar_texto, c["prompt"], c.get("sistema", ""),
                                 max_tokens=c.get("max_tokens", 512),
                                 temperatura=c.get("temperatura", 0.7), modelo=c.get("modelo"))
        return JSONResponse({"texto": texto})
    except Exception as e:
        return _erro(e)


@_rota("/api/imagem")
async def api_imagem(request: Request):
    """Devolve image/jpeg. Com "formato": "json" devolve base64."""
    c = await _corpo(request)
    try:
        dados = await _rodar(motor.gerar_imagem, c["prompt"], c.get("passos", 2),
                             c.get("guidance", 0.0), c.get("largura", 768), c.get("altura", 768),
                             c.get("seed", -1), c.get("negativo", ""), c.get("modelo"))
    except Exception as e:
        return _erro(e)
    if c.get("formato") == "json":
        return JSONResponse({"imagem_base64": base64.b64encode(dados).decode()})
    return Response(dados, media_type="image/jpeg")


@_rota("/api/video")
async def api_video(request: Request):
    c = await _corpo(request)
    try:
        nome = await _rodar(motor.gerar_video, c["prompt"], c.get("quadros"), c.get("passos"),
                            c.get("guidance"), c.get("largura"), c.get("altura"),
                            c.get("fps", 16), c.get("seed", -1), c.get("modelo"),
                            c.get("preset", ""))
        return JSONResponse({"url": _url_arquivo(nome)})
    except Exception as e:
        return _erro(e)


@_rota("/api/tarefa/{tarefa_id}", ("GET",))
async def api_tarefa(request: Request):
    return JSONResponse(motor.consultar(request.path_params["tarefa_id"]))


@_rota("/api/fala")
async def api_fala(request: Request):
    c = await _corpo(request)
    try:
        return Response(await _rodar(motor.falar, c["texto"], c.get("modelo")),
                        media_type="audio/wav")
    except Exception as e:
        return _erro(e)


@_rota("/api/transcrever")
async def api_transcrever(request: Request):
    """JSON {"audio": URL ou base64} ou multipart com campo "arquivo"."""
    try:
        if request.headers.get("content-type", "").startswith("multipart/"):
            form = await request.form()
            audio = base64.b64encode(await form["arquivo"].read()).decode()
            idioma, modelo = form.get("idioma"), form.get("modelo")
        else:
            c = await _corpo(request)
            audio, idioma, modelo = c["audio"], c.get("idioma"), c.get("modelo")
        return JSONResponse({"texto": await _rodar(motor.transcrever, audio, idioma, modelo)})
    except Exception as e:
        return _erro(e)


@_rota("/api/pipeline")
async def api_pipeline(request: Request):
    c = await _corpo(request)
    try:
        resultado = await _rodar(motor.executar_pipeline, c["tarefa"], c["repo_id"],
                                 c["entrada"], c.get("parametros"))
        return JSONResponse({"resultado": resultado})
    except Exception as e:
        return _erro(e)


@_rota("/arquivos/{nome}", ("GET",))
async def api_arquivo(request: Request):
    nome = os.path.basename(request.path_params["nome"])
    caminho = os.path.join(motor.PASTA_SAIDA, nome)
    if not os.path.isfile(caminho):
        return _erro(FileNotFoundError(nome), 404)
    return FileResponse(caminho)


# ================================================== API compatível com OpenAI
# Permite usar apps e bibliotecas feitas para OpenAI apontando base_url para {url}/{TOKEN}/v1

@_rota("/v1/models", ("GET",))
async def oa_modelos(request: Request):
    dados = [{"id": r, "object": "model", "owned_by": "local"} for r in motor.status()["modelos_carregados"]]
    return JSONResponse({"object": "list", "data": dados})


@_rota("/v1/chat/completions")
async def oa_chat(request: Request):
    c = await _corpo(request)
    modelo = c.get("model")
    if modelo in ("", "default", None) or "/" not in modelo:
        modelo = None
    mensagens = [{"role": m["role"], "content": m["content"] if isinstance(m["content"], str)
                  else " ".join(p.get("text", "") for p in m["content"])} for m in c["messages"]]
    try:
        texto = await _rodar(motor.conversar, mensagens, c.get("max_tokens") or 512,
                             c.get("temperature", 0.7), modelo)
    except Exception as e:
        return _erro(e)
    return JSONResponse({
        "id": f"chatcmpl-{uuid.uuid4().hex}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": modelo or motor.PADROES["texto"],
        "choices": [{"index": 0, "finish_reason": "stop",
                     "message": {"role": "assistant", "content": texto}}],
    })


@_rota("/v1/images/generations")
async def oa_imagens(request: Request):
    c = await _corpo(request)
    largura, altura = (int(x) for x in c.get("size", "768x768").split("x"))
    modelo = c.get("model") if "/" in (c.get("model") or "") else None
    try:
        imagens = [await _rodar(motor.gerar_imagem, c["prompt"], 2, 0.0, largura, altura, -1, "",
                                modelo) for _ in range(c.get("n", 1))]
    except Exception as e:
        return _erro(e)
    return JSONResponse({"created": int(time.time()),
                         "data": [{"b64_json": base64.b64encode(i).decode()} for i in imagens]})


@_rota("/v1/audio/speech")
async def oa_fala(request: Request):
    c = await _corpo(request)
    modelo = c.get("model") if "/" in (c.get("model") or "") else None
    try:
        return Response(await _rodar(motor.falar, c["input"], modelo), media_type="audio/wav")
    except Exception as e:
        return _erro(e)


@_rota("/v1/audio/transcriptions")
async def oa_transcricao(request: Request):
    form = await request.form()
    modelo = form.get("model") if "/" in (form.get("model") or "") else None
    try:
        audio = base64.b64encode(await form["file"].read()).decode()
        texto = await _rodar(motor.transcrever, audio, form.get("language"), modelo)
    except Exception as e:
        return _erro(e)
    return JSONResponse({"text": texto})


# ================================================================== início

def iniciar(ngrok_token: str | None = None, pre_carregar: list[str] | None = None) -> dict:
    """Sobe o servidor em segundo plano, abre o túnel ngrok e devolve as URLs.

    pre_carregar: tipos para já deixar na GPU, ex.: ["texto", "imagem"].
    """
    import threading

    threading.Thread(target=lambda: mcp.run(transport="streamable-http"), daemon=True).start()
    if ngrok_token:
        from pyngrok import conf, ngrok

        # Passa o token direto na configuração: set_auth_token falha nas versões novas do
        # ngrok quando o token já foi salvo numa execução anterior.
        conf.get_default().auth_token = ngrok_token
        ngrok.kill()  # fecha túneis de uma execução anterior desta sessão
        opcoes = {"domain": NGROK_DOMINIO} if NGROK_DOMINIO else {}
        estado["url_publica"] = ngrok.connect(PORTA, "http", **opcoes).public_url
    for tipo in pre_carregar or []:
        print(motor.carregar(None, tipo))
    base = f"{estado['url_publica']}{BASE}"
    urls = {"conector_claude": f"{base}/mcp", "api": f"{base}/api", "openai_base_url": f"{base}/v1"}
    print("\nNão compartilhe estas URLs, elas contêm o token de acesso:")
    for nome, url in urls.items():
        print(f"  {nome}: {url}")
    print()
    return urls


if __name__ == "__main__":
    iniciar(os.environ.get("NGROK_AUTHTOKEN"),
            [t for t in os.environ.get("PRE_CARREGAR", "").split(",") if t])
    while True:
        time.sleep(3600)


## 3. Configuração

In [ ]:
NGROK_AUTHTOKEN = "COLE_SEU_TOKEN_AQUI"

# Para a URL do conector continuar a mesma quando o Colab reiniciar:
MCP_TOKEN = ""       # invente um token longo, ex.: "meu-token-secreto-8f3k29dk"
NGROK_DOMINIO = ""   # seu domínio grátis do ngrok, ex.: "nome-aleatorio.ngrok-free.app"

# Tipos para já deixar carregados na GPU: "texto", "imagem", "fala", "transcrever".
# Para usar vídeo numa T4, deixe vazio: o vídeo precisa da GPU inteira.
PRE_CARREGAR = []

import os
os.environ["MCP_TOKEN"] = MCP_TOKEN
os.environ["NGROK_DOMINIO"] = NGROK_DOMINIO
os.environ["CACHE_VIDEO"] = "0.05"   # cache entre passos no vídeo (0 desliga)
os.environ["LORA_VIDEO"] = ""        # LoRA de destilação para vídeo (opcional)
os.environ["COMPILAR"] = "0"         # 1 = torch.compile nas imagens

## 4. Iniciar

In [ ]:
import servidor
urls = servidor.iniciar(NGROK_AUTHTOKEN, PRE_CARREGAR)

## 5. Teste rápido (opcional)

In [ ]:
import requests
from IPython.display import Image, display

print(requests.post(urls["api"] + "/texto", json={"prompt": "Diga olá em uma frase."}).json())
r = requests.post(urls["api"] + "/imagem", json={"prompt": "um gato astronauta, foto realista"})
display(Image(r.content))

## 6. Manter ativo
Deixe esta célula rodando enquanto usa o conector ou a API.

In [ ]:
import time
while True:
    time.sleep(60)